In [1]:
from datetime import datetime

# Print the date and time of the last run (European format)
print("Last run:", datetime.now().strftime("%d.%m.%Y %H:%M:%S"))

Last run: 06.03.2026 15:15:36



    title: Analyze
    author: Jean Cazalis
    date: Mar 5, 2026
    
---

**Description:** 

In [ ]:
import time

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from sym_contractions import Hamiltonian, build_observables

# Build or load the observables

In [24]:
###################################################################
# Parameters
###################################################################
interaction_labels = ["XXXX_p2341"]  # interaction terms
observable_labels = ["X_p1", "P_p1"]  # observables of interest
# Lambdas = [10, 12, 14, 16, 18, 20]  # cutoff
Lambdas = [10, 12, 14]  # cutoff
mass = 0.5
###################################################################

# Load or build and save observables, then build Hamiltonian for each Lambda
compute_time = []
ham_dict = {}
obs_dict = {}
K = len(interaction_labels)
for Lambda in Lambdas:
    print(f"Processing Lambda={Lambda}...")
    start_time = time.time()
    observables = build_observables(
        interaction_labels + observable_labels,
        Lambda=Lambda,
        mass=mass,
        clean=True,
        verbose=False,
        save=True,
    )
    ham_dict[Lambda] = Hamiltonian(
        observables=observables[:K], Lambda=Lambda, default_mass=mass
    )
    obs_dict[Lambda] = observables[K:]
    compute_time.append(time.time() - start_time)

Processing Lambda=10...
Processing Lambda=12...
Processing Lambda=14...


# Compute the expectation value in the observable

In [ ]:
Lambda = 10
d = 7
mass = 2


def coupling(d, m):
    return m**2 / d


ham = ham_dict[Lambda]
obs1 = obs_dict[Lambda][0]
obs2 = obs_dict[Lambda][1]

# Evaluate the Hamiltonian and the observable in d and m
H = ham.evaluate(d=d, coupling=coupling, mass=mass, ground_state_only=False)
obs1_eval = obs1.evaluate(d=d, mass=mass, ground_state_only=False)
obs2_eval = obs2.evaluate(d=d, mass=mass, ground_state_only=False)


# Compute the ground state
eigvals, eigstates = np.linalg.eigh(H)
print(
    f"Normalized ground state energy for Lambda={Lambda}, d={d}: {eigvals[0] / d**2:.6f}"
)

psi = eigstates[:, 0]
print(f"\nGround state: {psi}")

# Some sanity checks
assert np.isclose(np.linalg.norm(psi), 1), "Ground state is not normalized"
assert np.isclose(psi.conj().T @ H @ psi, eigvals[0]), "Ground state energy mismatch"


# Compute the expectation value in the observable in the ground state
obs1_eval.expectation_value(d, state=psi)
print(
    f"\nExpectation value of the observable {obs1_eval.label} in the ground state: {obs1_eval.expectation_value(d, state=psi):.6f}"
)

obs2_eval.expectation_value(d, state=psi)
print(
    f"\nExpectation value of the observable {obs2_eval.label} in the ground state: {obs2_eval.expectation_value(d, state=psi):.6f}"
)

Normalized ground state energy for Lambda=10, d=7: 1.304271

Ground state: [-5.76709029e-01  0.00000000e+00  4.77405525e-01 -3.83496742e-01
  6.66133815e-16  3.05311332e-16 -2.77555756e-16 -2.15141858e-01
  1.36261789e-01 -3.16382986e-01  1.66329410e-01 -1.05562065e-01
 -3.21791205e-16 -1.55040911e-16 -2.09684700e-16  2.56793285e-16
 -6.57026517e-17  4.12742215e-17 -4.25989810e-17  5.76595962e-02
 -2.38064906e-02  1.41358697e-01 -1.04017514e-01 -1.11488423e-01
  8.37000901e-19  5.36859781e-02  1.08989053e-01 -8.61100213e-02
  2.55241697e-02 -1.10304526e-02  8.47353652e-17  3.40089274e-17
  1.36658193e-16 -1.15228687e-16  5.51133433e-17 -1.26153188e-18
 -2.63377238e-17 -4.56508133e-17  8.91683698e-17 -3.96432960e-17
  4.17291461e-17 -3.71675284e-18 -1.44314121e-17  5.65259904e-18
 -2.57546286e-18 -3.40745317e-03 -3.90232963e-03 -3.71764099e-02
  3.42281546e-02  1.93433767e-02 -2.25140543e-19 -1.32437800e-02
 -4.83825351e-02  2.33219075e-02 -6.56339398e-02  3.70542315e-02
 -1.75194202e-0

In [38]:
eigvals, eigstates = np.linalg.eigh(H)
np.allclose(eigstates.conj().T @ H @ eigstates, np.diag(eigvals))

t1 = 2.0 * 1j
t2 = 4.0 * 1j

Ht1 = eigstates @ np.diag(np.exp(1j * t1 * eigvals)) @ eigstates.conj().T
Ht2 = eigstates @ np.diag(np.exp(1j * t2 * eigvals)) @ eigstates.conj().T

print(f"Expectation value of operator {obs1_eval.label}")
print(obs1_eval.expectation_value(d, state=Ht1))
print(obs1_eval.expectation_value(d, state=Ht2))
print()
print(f"Expectation value of operator {obs2_eval.label}")
print(obs2_eval.expectation_value(d, state=Ht1))
print(obs2_eval.expectation_value(d, state=Ht2))

Expectation value of operator X_p1
4.6734845539563136e-71
1.4398545819591294e-126

Expectation value of operator P_p1
0.0
0.0


# Compute Spectra And Partition Function

In [29]:
gMax = 1.0
dList = [2, 3, 4, 5, 6, 7]
mass = 2.0  # normalized GS energy for free theory is 1
Tmax = 10.0

gg = np.arange(0.0, gMax + 0.01, 0.1)
TT = np.arange(0.1, Tmax + 0.1, 0.1)
betas = 1.0 / TT


def coupling(g, d, m):
    return m**2 * g / d


spectra_dict = {}
partition_dict = {}

for Lambda in Lambdas:
    ham = ham_dict[Lambda]
    print(f"Computing spectra and partition function for Lambda={Lambda}...")

    for d in dList:
        # Z(g, beta) grid for this (Lambda, d)
        Z_grid = np.zeros((len(betas), len(gg)), dtype=np.float64)

        for ig, g in enumerate(gg):
            g_eval = g

            def g_of_dm(d_val, m_val, _g=g_eval):
                return coupling(_g, d_val, m_val)

            H = ham.evaluate(d=d, coupling=g_of_dm, mass=mass, ground_state_only=False)
            eigvals = np.linalg.eigvalsh(H).astype(np.float64)
            spectra_dict[(Lambda, d, g_eval)] = eigvals / d**2

            Z_grid[:, ig] = np.exp(-betas[:, None] * eigvals[None, :]).sum(axis=1)

        partition_dict[(Lambda, d)] = Z_grid

print("Done.")
print(f"Stored spectra keys: {len(spectra_dict)}")
print(f"Stored partition grids: {len(partition_dict)}")

Computing spectra and partition function for Lambda=10...
Computing spectra and partition function for Lambda=12...
Computing spectra and partition function for Lambda=14...


/var/folders/ss/5s7f777x65d9g2d3ft1dcs1c0000gp/T/ipykernel_11389/4194418005.py:36: RuntimeWarning: overflow encountered in exp
  Z_grid[:, ig] = np.exp(-betas[:, None] * eigvals[None, :]).sum(axis=1)


Computing spectra and partition function for Lambda=16...
Computing spectra and partition function for Lambda=18...
Computing spectra and partition function for Lambda=20...
Done.
Stored spectra keys: 396
Stored partition grids: 36


# Interactive Contour Plot Of Partition Function

In [ ]:
def plot_partition_contour(Lambda, d):
    Z = partition_dict[(Lambda, d)]  # shape (n_beta, n_g)
    G, B = np.meshgrid(gg, betas)

    fig, ax = plt.subplots(figsize=(8, 5))
    contour = ax.contourf(G, B, Z, levels=40, cmap="viridis")
    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_label(r"$Z(\beta, g)$")

    ax.set_xlabel(r"$g$", fontsize=12)
    ax.set_ylabel(r"$\beta$", fontsize=12)
    ax.set_title(rf"Partition function $Z(\beta,g)$ for $\Lambda={Lambda}$, $d={d}$")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()


lambda_widget = widgets.Dropdown(
    options=Lambdas,
    value=Lambdas[0],
    description="Lambda:",
    layout=widgets.Layout(width="250px"),
)

d_widget = widgets.Dropdown(
    options=dList,
    value=dList[0],
    description="d:",
    layout=widgets.Layout(width="250px"),
)

ui = widgets.HBox([lambda_widget, d_widget])
out = widgets.interactive_output(
    plot_partition_contour,
    {"Lambda": lambda_widget, "d": d_widget},
)
display(ui, out)

Output()

# Quick Check: Example Spectrum Slice

In [ ]:
Lambda = Lambdas[0]
d = dList[0]
g = gg[0]

print(f"Example key: (Lambda={Lambda}, d={d}, g={g:.2f})")
print("First 10 eigenvalues:")
print(spectra_dict[(Lambda, d, g)][:10])